# Sudoku Knowledge Representation & Inference

You will implement three functions -- `build_general_kb`, `build_definite_kb`, `pl_bc_entails` -- and the full-grid solving logic in `sudoku_solver.py`.

This notebook imports and tests those functions. The Streamlit app (`sudoku_app.py`) must import the same implementation from `sudoku_solver.py`; do not copy or rewrite the solver functions inside the app.

**Rules:**

- In `sudoku_solver.py`, import only from `utils.py` and `logic_.py`; do not modify either file.
- Do not duplicate the core solver functions in this notebook or `sudoku_app.py`.
- All other content in this notebook may be edited freely.


In [1]:
from utils import *
from logic_ import *
import json
import time
import importlib
import sudoku_solver

# Reload so edits made to sudoku_solver.py are picked up when this cell is rerun.
importlib.reload(sudoku_solver)

from sudoku_solver import (
    atom,
    build_general_kb,
    build_definite_kb,
    solve_full_grid_fc,
    pl_bc_entails,
    solve_full_grid_bc,
)


## Loading a puzzle from JSON

Puzzles are provided as JSON, not embedded in this notebook. Each file looks like:

```json
{
  "n": 9, "box_h": 3, "box_w": 3,
  "puzzles": [
    {
      "givens": {"1_1": 3, "2_3": 1, ...},
      "given_count": 28,
      "solution": {"1_1": 3, "1_2": 4, ...}
    },
    ...
  ]
}
```

`"r_c"` string keys map to the value at row `r`, column `c` (1-indexed). `given_count` is exactly how many cells are given. `solution` is included so you can check your own work as you go, but your functions must not read `solution` to answer a query, they should only read `givens`.

In [2]:
#do not change this function, it is used to load the puzzle pool from a json file
def load_pool(path):
    with open(path) as f:
        raw = json.load(f)
    puzzles = []
    for p in raw['puzzles']:
        givens = {tuple(int(x) for x in k.split('_')): v for k, v in p['givens'].items()}
        solution = {tuple(int(x) for x in k.split('_')): v for k, v in p['solution'].items()}
        puzzles.append({'givens': givens, 'solution': solution, 'given_count': p['given_count']})
    return raw['n'], raw['box_h'], raw['box_w'], puzzles

n, box_h, box_w, puzzle_pool = load_pool('puzzles.json')
print(f'{len(puzzle_pool)} puzzles loaded, {n}x{n} grid, {box_h}x{box_w} boxes')
print('given_count values:', sorted(p['given_count'] for p in puzzle_pool))

# Pick one puzzle to work with through the rest of this notebook.
puzzle = puzzle_pool[0]
givens = puzzle['givens']
print(f"working puzzle has {puzzle['given_count']} givens")
for r in range(1, n + 1):
    print([givens.get((r, c), '.') for c in range(1, n + 1)])

5 puzzles loaded, 9x9 grid, 3x3 boxes
given_count values: [30, 33, 36, 39, 42]
working puzzle has 30 givens
['.', 3, '.', '.', '.', '.', '.', '.', '.']
[2, '.', '.', '.', '.', 7, 8, 6, '.']
[5, 8, '.', 2, 6, '.', '.', 3, '.']
[7, 5, '.', '.', '.', '.', '.', 8, '.']
['.', '.', '.', '.', 7, '.', 5, '.', 4]
['.', '.', '.', 5, 3, '.', '.', 9, 6]
['.', 1, 2, '.', '.', 9, '.', '.', '.']
[6, 4, '.', '.', 5, 8, 9, '.', '.']
['.', '.', '.', '.', 2, 3, '.', '.', '.']


Notice: `pl_fc_entails`/`pl_resolution`/`tt_entails` are all given to you, already implemented, in `logic_.py`. The one algorithm you write yourself in this assignment is **backward chaining** (`pl_bc_entails`) -- see Task 2.

## Define Symbols

Two families of propositional symbols, for row $r$, column $c$, value $v$ (all ranging over $1$ to $n$):

| Symbol | Meaning |
|---|---|
| $\mathit{Is}_{rcv}$ | cell $(r,c)$ has value $v$ |
| $\mathit{Not}_{rcv}$ | cell $(r,c)$ does **not** have value $v$ |

For each representation in Task 1, determine which of these two families is actually needed.

Hint: consider what form a definite (Horn) clause must take, and what `PropDefiniteKB.tell()` will accept.

### Helper function

`atom(prefix, r, c, v)`, where prefix can be either `Is` or `Not`, is a naming helper so propositional symbols need not be typed. It is provided for you in `sudoku_solver.py`; do not change it.

In code, $\mathit{Is}_{rcv}$ and $\mathit{Not}_{rcv}$ are written as single-word symbol names, e.g. `Is3_2_4` and `Not3_2_4`: the prefix is followed immediately by `r`, then by `c` and `v` separated by underscores. No separator is needed between the prefix and `r` since `expr()` only requires a symbol name to start with an uppercase letter. So `Is3_2_4` is parsed as one valid symbol.


In [3]:
# atom() is provided in sudoku_solver.py and imported above.


## Part A: Design the Sudoku Solver

### A.1) Knowledge Representation - Build KB

Every well-posed Sudoku puzzle satisfies exactly these conditions:

- Each cell is assigned **at least one** value from $\{1, \dots, n\}$.
- Each cell is assigned **at most one** value from $\{1, \dots, n\}$, i.e., it cannot hold two different values at once.
- No two cells in the same row hold the same value.
- No two cells in the same column hold the same value.
- No two cells in the same box hold the same value.
- The **givens** cells hold their stated values.

Formalize these Sudoku constraints as propositional logic, in **two** representations:

**(a) General clauses -  `build_general_kb`.**

Encode each of the following directly: no restriction here; you may use arbitrary disjunctions of positive or negated literals. Must return a `PropKB`. 

**(b) Definite (Horn) clauses:- `build_definite_kb`.** Must return a `PropDefiniteKB`. Recall a definite clause is a disjunction of literals with exactly one *positive* literal.

Equivalently written as an implication whose conclusion is a single positive literal and whose premises are a conjunction of positive literals: `P1 & P2 & ... & Pk ==> Q`.

`PropDefiniteKB.tell()` will reject anything else.

Both functions take the puzzle's givens as fixed facts.

- **Implement `build_general_kb()` and `build_definite_kb()` in sudoku_solver.py**
- **Rerun the import cell near the top of this notebook after making changes.**
- **Explain your representation in Conceptual Question 1 below.**

In [4]:
# Implement build_general_kb() and build_definite_kb() in sudoku_solver.py.
# Rerun the import cell near the top of this notebook after making changes.


### A.2) Solve the Puzzle

**(a) Resolution and model checking on the general representation.** Using `build_general_kb` and the library's `pl_resolution` / `tt_entails`, try to solve the puzzle, i.e., for each cell, determine which value is entailed. Attempt this in the code cell below: it is provided commented out, because both are sound and complete on `build_general_kb`'s output but neither scales to the full grid, and it is expected to hang or take an impractically long time. Uncomment a few lines at a time and give each at most about 30 seconds; use Kernel > Interrupt if it hasn't returned by then, and note what you observed.

Explain in your own words: what specifically makes `pl_resolution`'s cost grow out of control here, and separately, what makes `tt_entails`'s cost grow out of control? A simple complexity argument for each is the expected answer.

**Use your observations in Conceptual Question 2 below.**


In [5]:
# Run the supplied algorithms in disposable child processes. This changes
# neither the KB encoding nor either inference algorithm.
import subprocess
import sys
import psutil


def bounded_general_query(method, size, givens_for_test,
                          timeout_seconds=30, memory_limit_mb=256):
    """Return an observed verdict or a resource-limit result, never a fake False."""
    import json
    payload = json.dumps({
        'method': method,
        'dimensions': size,
        'givens': {f'{r}_{c}': v for (r, c), v in givens_for_test.items()},
    })
    worker = r"""
import json, sys, time
from sudoku_solver import atom, build_general_kb
from logic_ import pl_resolution, tt_entails, associate
p = json.loads(sys.argv[1])
g = {tuple(map(int, k.split('_'))): v for k, v in p['givens'].items()}
n, h, w = p['dimensions']
kb = build_general_kb(n, h, w, g)
query = atom('Is', 1, 1, g.get((1, 1), 1))
print(json.dumps({'clauses': len(kb.clauses), 'query': str(query)}), flush=True)
start = time.perf_counter()
if p['method'] == 'resolution':
    verdict = pl_resolution(kb, query)
else:
    verdict = tt_entails(associate('&', kb.clauses), query)
print(json.dumps({'verdict': verdict, 'inference_seconds': time.perf_counter() - start}), flush=True)
"""
    start = time.perf_counter()
    process = subprocess.Popen(
        [sys.executable, '-B', '-u', '-c', worker, payload],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
    )
    monitored = psutil.Process(process.pid)
    status = 'completed'
    peak_rss = 0
    try:
        while process.poll() is None:
            try:
                rss = monitored.memory_info().rss
            except psutil.NoSuchProcess:
                break
            peak_rss = max(peak_rss, rss)
            if rss > memory_limit_mb * 1024 ** 2:
                status = 'memory_limit'
                process.kill()
                break
            if time.perf_counter() - start >= timeout_seconds:
                status = 'time_limit'
                process.kill()
                break
            time.sleep(0.02)
        stdout, stderr = process.communicate(timeout=5)
    finally:
        if process.poll() is None:
            process.kill()
            process.communicate(timeout=5)
    messages = [json.loads(line) for line in stdout.splitlines() if line.strip()]
    if status == 'completed' and process.returncode != 0:
        raise RuntimeError(stderr or f'Worker exited with {process.returncode}')
    result = {
        'method': method,
        'grid': f'{size[0]}x{size[0]}',
        'status': status,
        'wall_seconds': round(time.perf_counter() - start, 3),
        'sampled_peak_rss_mb': round(peak_rss / 1024 ** 2, 1),
        'limits': f'{timeout_seconds}s / {memory_limit_mb} MiB sampled RSS',
    }
    for message in messages:
        result.update(message)
    return result


# Sanity check: these same implementations can finish on a tiny puzzle.
small_experiments = [
    bounded_general_query(method, (2, 1, 2), {(1, 1): 1})
    for method in ('resolution', 'truth_table')
]
for result in small_experiments:
    assert result['status'] == 'completed' and result['verdict'] is True
    print(result)

# Full-size attempts use givens only. A limit result is not an entailment verdict.
# RSS is sampled, so allocation can overshoot the threshold between samples;
# the resolution worker is stopped before it can build its enormous pair list.
general_experiments = [
    bounded_general_query(method, (n, box_h, box_w), givens)
    for method in ('resolution', 'truth_table')
]
for result in general_experiments:
    print(result)

{'method': 'resolution', 'grid': '2x2', 'status': 'completed', 'wall_seconds': 0.103, 'sampled_peak_rss_mb': 46.6, 'limits': '30s / 256 MiB sampled RSS', 'clauses': 21, 'query': 'Is1_1_1', 'verdict': True, 'inference_seconds': 0.0008830000297166407}
{'method': 'truth_table', 'grid': '2x2', 'status': 'completed', 'wall_seconds': 0.101, 'sampled_peak_rss_mb': 44.9, 'limits': '30s / 256 MiB sampled RSS', 'clauses': 21, 'query': 'Is1_1_1', 'verdict': True, 'inference_seconds': 0.0006904579931870103}


{'method': 'resolution', 'grid': '9x9', 'status': 'memory_limit', 'wall_seconds': 0.502, 'sampled_peak_rss_mb': 267.3, 'limits': '30s / 256 MiB sampled RSS', 'clauses': 11775, 'query': 'Is1_1_1'}
{'method': 'truth_table', 'grid': '9x9', 'status': 'time_limit', 'wall_seconds': 30.001, 'sampled_peak_rss_mb': 67.0, 'limits': '30s / 256 MiB sampled RSS', 'clauses': 11775, 'query': 'Is1_1_1'}


### Observation notes

Record what you observed from the resolution/model-checking experiment here. Use these observations when answering Conceptual Question 2 below.


The output above records the exact query, knowledge-base size, resource limit,
observed status, elapsed wall time and sampled peak resident memory for each run.
The 2x2 experiments verify that both supplied procedures can answer a small query.
The 9x9 runs are separate processes with a 30-second wall-time limit and a sampled
256 MiB resident-memory threshold. A `time_limit` or `memory_limit` result means
that this protected attempt did not finish; it does **not** mean that the query is
False. Wall time includes process startup and KB construction. The memory
threshold is sampled rather than a strict operating-system allocation cap.

**(b) Forward chaining on the full grid --** Implement `solve_full_grid_fc()` in sudoku_solver.py. Using your above `build_definite_kb` and the library's `pl_fc_entails` (no need to reimplement them), solve the whole puzzle. Find the value that's entailed for every cell. Reconstruct and display the solved grid.

**(c) Backward chaining -- implement it yourself.**
```python
pl_bc_entails(kb, query) -> bool
```
Implement `pl_bc_entails()` in sudoku_solver.py. Start from the query and recursively try to prove each premise of a rule whose conclusion matches the current goal, bottoming out at known facts. Your function must agree with `pl_fc_entails` on every cell/value pair in this puzzle including correctly returning `False` for values that are *not* part of the solution.

**(d) Backward chaining on the full grid --** Implement `solve_full_grid_bc` in sudoku_solver.py. Using your above `build_definite_kb` and your own `pl_bc_entails`, solve the whole puzzle the same way `solve_full_grid_fc` does: for every cell, try each candidate value until `pl_bc_entails` confirms one. Time both `solve_full_grid_fc` and `solve_full_grid_bc` on the same puzzle and compare. Use your measured result where relevant in Conceptual Question 5.


In [6]:
# Implement solve_full_grid_fc(), pl_bc_entails(), and solve_full_grid_bc()
# in sudoku_solver.py. Rerun the import cell near the top after making changes.


### Verifying the algorithms

Uncomment the following block of code to validate your code.

In [7]:
# Time each original full-grid entry point, including its KB and index creation.
t0 = time.perf_counter()
solved = solve_full_grid_fc(n, box_h, box_w, givens)
fc_time = time.perf_counter() - t0
assert solved == puzzle['solution']

definite_kb = build_definite_kb(n, box_h, box_w, givens)
query_count = 0
for r in range(1, n + 1):
    for c in range(1, n + 1):
        for v in range(1, n + 1):
            expected = puzzle['solution'][(r, c)] == v
            assert pl_bc_entails(definite_kb, atom('Is', r, c, v)) == expected
            query_count += 1

t0 = time.perf_counter()
solved_bc = solve_full_grid_bc(n, box_h, box_w, givens)
bc_time = time.perf_counter() - t0
assert solved_bc == puzzle['solution']
assert solved_bc == solved

for label, grid in [('Forward chaining', solved), ('Backward chaining', solved_bc)]:
    print(f'{label}: solved grid')
    for r in range(1, n + 1):
        print(' '.join(str(grid[(r, c)]) for c in range(1, n + 1)))
    print()

print(f'All {query_count} cell/value queries passed, including incorrect values.')
print(f'solve_full_grid_fc: {fc_time:.2f}s')
print(f'solve_full_grid_bc: {bc_time:.2f}s')
print('BENCHMARK_JSON=' + json.dumps({
    'given_count': len(givens), 'fc_seconds': fc_time, 'bc_seconds': bc_time,
}))

Forward chaining: solved grid
1 3 6 9 8 5 4 7 2
2 9 4 3 1 7 8 6 5
5 8 7 2 6 4 1 3 9
7 5 1 4 9 6 2 8 3
3 6 9 8 7 2 5 1 4
4 2 8 5 3 1 7 9 6
8 1 2 6 4 9 3 5 7
6 4 3 7 5 8 9 2 1
9 7 5 1 2 3 6 4 8

Backward chaining: solved grid
1 3 6 9 8 5 4 7 2
2 9 4 3 1 7 8 6 5
5 8 7 2 6 4 1 3 9
7 5 1 4 9 6 2 8 3
3 6 9 8 7 2 5 1 4
4 2 8 5 3 1 7 9 6
8 1 2 6 4 9 3 5 7
6 4 3 7 5 8 9 2 1
9 7 5 1 2 3 6 4 8

All 729 cell/value queries passed, including incorrect values.
solve_full_grid_fc: 12.67s
solve_full_grid_bc: 5.45s
BENCHMARK_JSON={"given_count": 30, "fc_seconds": 12.670773999998346, "bc_seconds": 5.453981541970279}


## Part B: Conceptual Questions

Answer all five questions directly in this notebook. Replace each **Your answer:** placeholder with your own response.


### 1. Detailed Representation Strategy: General vs. Definite (Horn) Encoding

Explain in detail how you formalized the Sudoku puzzle constraints into propositional logic across both Knowledge Base representations:

**(a) General KB Strategy (`build_general_kb`):** Detail how standard Sudoku rules (e.g., at-least-one value per cell, at-most-one value per cell, row/column/box uniqueness) are directly translated into Conjunctive Normal Form (CNF) clauses without structural restrictions.

**(b) Definite KB Strategy (`build_definite_kb`):** Definite/Horn clauses strictly permit at most one positive literal per clause, prohibiting disjunctive constraints like $(Is_{r,c,1} \lor Is_{r,c,2} \lor Is_{r,c,3} \lor ... \lor Is_{r,c,n})$. Explain step-by-step how your encoding deals with this issue.


### (a) General KB

In `build_general_kb`, each possible cell/value assignment is represented by a propositional atom `Is_r_c_v`, meaning that cell `(r,c)` contains value `v`.

Because `PropKB` permits general propositional clauses, the main Sudoku constraints can be represented directly.

For each cell, the requirement that it contains at least one value is encoded as:

$$
Is_{r,c,1} \lor Is_{r,c,2} \lor \cdots \lor Is_{r,c,n}.
$$

The requirement that a cell contains at most one value is encoded pairwise. For any two distinct values $v_1$ and $v_2$:

$$
\neg Is_{r,c,v_1} \lor \neg Is_{r,c,v_2}.
$$

Row, column, and box uniqueness constraints use the same pairwise idea. For example, two different cells in the same row cannot both contain the same value $v$:

$$
\neg Is_{r,c_1,v} \lor \neg Is_{r,c_2,v}.
$$

Equivalent clauses are generated for columns and boxes. Each given value is then added directly to the KB as a fact of the form `Is_r_c_v`.

### (b) Definite / Horn KB

The definite-clause representation requires a different encoding because a definite clause has exactly one positive conclusion. In particular, the direct "at least one value" disjunction used in the general KB cannot be represented in the same form.

The definite KB therefore uses two kinds of positive propositional atoms:

- `Is_r_c_v`: cell `(r,c)` is known to contain value `v`
- `Not_r_c_v`: value `v` has been eliminated from cell `(r,c)`

The first group of rules performs elimination. For example, if a cell is known to contain 5, another cell in the same row cannot contain 5:

$$
Is_{1,1,5} \Rightarrow Not_{1,2,5}.
$$

Similar rules eliminate alternative values from the same cell and eliminate the same value from cells in the same row, column, and box.

The second group consists of last-candidate rules. If all values except one have been eliminated from a cell, the remaining value can be concluded. For example:

$$
Not_{1,1,1} \land Not_{1,1,2} \land \cdots \land Not_{1,1,8}
\Rightarrow Is_{1,1,9}.
$$

The givens are inserted as initial `Is` facts. Forward or backward chaining can then derive additional `Not` and `Is` facts by repeatedly applying these definite rules.

`Not_r_c_v` is an independent positive atom, not Python/logical negation of `Is_r_c_v`. Its elimination meaning comes from the rules. This Horn KB encodes a sound elimination procedure for valid Sudoku puzzles; it is not a model-equivalent replacement of the complete general constraint KB. Completeness of Horn inference means finding everything entailed by these rules, not solving every uniquely solvable Sudoku.


### 2. Theoretical Completeness vs. Computational Tractability

Model checking and resolution-refutation are sound and complete—they are guaranteed to terminate with a correct answer for any propositional KB. Despite this guarantee, explain whether you would use either as the default algorithm for solving Sudoku puzzles. *(Hint: Consider space/time complexity and state-space growth, and use your observations from the experiment above where relevant.)*


I would not use these particular truth-table and unrestricted resolution
implementations as the default solvers for the full Sudoku encoding.

For truth-table checking there are $9^3 = 729$ `Is` atoms. Enumerating Boolean
assignments has a worst-case search space of $2^{729}$, with additional work to
evaluate the KB for each assignment. Sudoku constraints reduce the satisfying
models, but the supplied truth-table procedure does not exploit them as a
specialized Sudoku or modern SAT solver would.

The supplied `pl_resolution` has a different bottleneck: it first materializes
all pairs of current clauses, then generates resolvents and repeats as the clause
set grows. The 30-given puzzle produces 11,775 clauses. Including the negated
query, the initial pair list alone contains
$11,776\times11,775/2 = 69,331,200$ pairs. Pair generation costs quadratic time and
space in the current clause count; later resolvents can cause further
combinatorial growth. The general encoding also contains redundant clauses from
overlapping row/column/box constraints, which add cost without changing meaning.

The experiment cell records actual outcomes under the stated time and memory
limits. The tiny 2x2 runs serve as a sanity check; the full-size runs show what
these supplied implementations can achieve within those bounds. Resource-limited
runs must not be reported as logical False or as evidence of unsoundness.

Soundness and completeness concern correctness and eventual answers in the
finite propositional setting, not practical time or memory. Horn elimination is
more practical for the supplied puzzles. This conclusion concerns the provided
educational algorithms and encoding, not every industrial SAT-solving approach.

### 3. Backward Chaining: Design, Pseudocode, and Challenges

Write pseudocode for `pl_bc_entails(kb, query)`, the backward-chaining algorithm you implemented. Show, at a level of detail that reveals the algorithm's structure (not full Python), how the function checks whether the query is already a known fact, finds candidate rules whose conclusion matches the current goal, recursively proves each premise of such a rule, and combines results—both across the premises of one rule and across multiple candidate rules—to reach a single boolean answer.

Then, in your own words, discuss the design challenges you had to work through to make your algorithm both correct and guaranteed to terminate on every puzzle, and explain how your pseudocode addresses them.


The implemented backward-chaining procedure is goal-driven: it begins with the query and searches backwards for rules whose conclusions could establish that query.

### Pseudocode

```text
BC-ENTAILS(KB, query):

    collect the known facts from KB
    index the rules by their conclusion
    proven = known facts

    repeat:
        record the current number of proven facts
        memo = empty

        if PROVE(query, empty visiting set, memo):
            return True

        if no new facts were proved:
            return False


PROVE(goal, visiting, memo):

    if goal is already proven:
        return True

    if goal has a memoized result:
        return the memoized result

    if goal is already in visiting:
        return False

    add goal to visiting

    for each rule whose conclusion is goal:

        all_premises_proved = True

        for each premise of the rule:
            if PROVE(premise, visiting, memo) is False:
                all_premises_proved = False
                break

        if all_premises_proved:
            remove goal from visiting
            add goal to proven
            memoize goal as True
            return True

    remove goal from visiting
    memoize goal as False
    return False
```

Within a single definite rule, the premises are combined using **AND**, because every premise must be established before the conclusion follows. For example:

$$
A \land B \Rightarrow Q
$$

requires both $A$ and $B$ to be proved.

By contrast, if several different rules have the same conclusion, those rules provide alternative proof paths and therefore behave as **OR** alternatives. For example:

$$
A \land B \Rightarrow Q
$$

and

$$
C \land D \Rightarrow Q
$$

allow $Q$ to be established through either $(A \land B)$ or $(C \land D)$.

To avoid infinite recursion caused by cyclic dependencies, the implementation maintains a `visiting` set containing the goals on the current recursive proof path. If a goal is encountered again while it is already in `visiting`, that proof path is stopped.

A `memo` dictionary is also used to avoid repeatedly solving the same subgoal during one proof attempt. Successfully proved subgoals are retained in a `proven` set. Because the KB contains only finitely many atoms, each retry either adds at least one new proved fact or terminates when no further progress is possible. This provides a finite stopping condition for the implemented procedure.

The implementation indexes rules by conclusion and reuses that index only while a snapshot of `kb.clauses` is unchanged. Adding, retracting, or replacing clauses rebuilds the facts and rule index. The `proven` set and proof trace belong to one query. A failed subgoal is memoized only for one attempt: failure caused by a cyclic path is not a permanent logical negation. When an attempt derives new facts, the next attempt clears its failure memo and retries. Each retry either enlarges the finite proven set or stops; a cycle with no fact-supported derivation never establishes itself.


### 4. Expressive Limits of Horn Logic

Named elimination techniques such as Naked Pairs and X-Wing can, in fact, be encoded as definite clauses, using the same `Is`/`Not` vocabulary as your `build_definite_kb`. Work out how you would encode one of these techniques as definite clauses, and discuss the consequences of doing so.


I choose **Naked Pairs**, using only the existing `Is` / `Not` vocabulary.
This is a conceptual extension; it is not required by the current basic solver.

Let $D=\{1,\ldots,9\}$. Choose two distinct cells $A=(r_A,c_A)$ and
$B=(r_B,c_B)$ in the same row, column or box, and two distinct values $x,y\in D$.
Let $C=(r_C,c_C)$ be any other cell in that unit. Use the following conjunction
as an abbreviation for the premises, not as a new propositional atom:

$$
\Gamma = \bigwedge_{v\in D\setminus\{x,y\}}
\left(Not_{r_A,c_A,v}\land Not_{r_B,c_B,v}\right).
$$

For every such choice of cells and values, add **two** definite clauses:

$$
\Gamma \Rightarrow Not_{r_C,c_C,x},
\qquad
\Gamma \Rightarrow Not_{r_C,c_C,y}.
$$

After expanding $\Gamma$, each implication has 14 positive `Not` atoms as
premises and one positive `Not` atom as its conclusion. Its CNF form therefore
has exactly one positive literal. No `Pair` or candidate-presence atom is needed.

For example, if cells (1,1) and (1,2) have both eliminated 1, 2, 4, 5, 6, 8 and 9,
then they are restricted to {3,7}. The conjunction of those fourteen elimination
facts can separately imply `Not1_3_3` and `Not1_3_7`, and analogous eliminations
for every other cell in row 1.

Why this is sound for a valid Sudoku: each of A and B must have a value, all
values outside {x,y} have been excluded, and their values must differ because
they share a unit. They therefore occupy x and y between them. Every other cell
in that unit must exclude both. This does **not** infer candidate membership
from the absence of a `Not` fact. Positive evidence excluding the other digits
is sufficient. It still works if one of the pair cells has already been solved.

These rules can unlock further last-candidate deductions and allow the Horn KB
to solve puzzles that basic elimination alone cannot. The tradeoff is rule size:
a naive 9x9 grounding enumerates 27 units, $\binom{9}{2}$ cell pairs,
$\binom{9}{2}$ value pairs, seven other cells and two conclusions, giving
489,888 extra rule instances before deduplication. Each has fourteen premises.
This increases construction time, memory and inference work. It extends the
available reasoning techniques without making a finite collection of such
techniques a universal Sudoku solver.

### 5. Data-Driven vs. Goal-Driven Performance

Forward chaining (data-driven) and backward chaining (goal-driven) are both sound and complete for Horn KBs, but their execution runtimes vary depending on the target query.

**(a)** Describe a scenario—in terms of total KB size versus the query-relevant subset—where backward chaining is significantly faster than forward chaining.

**(b)** Describe a scenario where backward chaining offers no performance advantage, or performs worse than forward chaining.

Use your measured forward- vs. backward-chaining result where relevant.


### (a) When backward chaining can be faster

If a large KB contains many unrelated rules but a query has only a small
relevant dependency subgraph, backward chaining can focus on that subgraph.
Forward chaining may derive unrelated consequences before reaching the target.
Rule indexing matters: even a focused query can pay a one-time cost to index a
large KB.

In this notebook's saved run on the 30-given puzzle, FC took **12.67 seconds** and BC took **5.45 seconds**. The FC/BC time ratio was **2.32**. Both calls include construction of a fresh KB, indexing and all candidate queries; module import and result validation are excluded. There is no cross-solver reuse of derived facts. These are single-run measurements on this machine, not a universal ranking of the algorithms.

The measured difference is specific to this puzzle and these implementations.
The FC solver repeatedly calls the supplied entailment procedure for candidate
values, rebuilding its per-query counters. It does index premises to avoid
repeated full-KB scans. BC indexes conclusions, keeps that index while the KB is
unchanged, and searches for a proof of each candidate; its proved facts are not
shared between different queries. These implementation details, as well as
query relevance, affect the result. This is not a claim that BC is always faster.

### (b) When backward chaining may offer no advantage

If a target depends on most of the KB, backward search avoids little work.
Many related queries can also repeatedly explore the same dependencies. A
forward-chaining implementation that computes and retains the full closure once
can then serve many queries cheaply; that is different from the per-candidate
FC strategy required and used here. Cyclic dependencies and unsuccessful proof
attempts can add BC work. The best choice depends on the KB, workload, indexing
and what results are reused.

## Part C: Streamlit Integration

Wrap your Sudoku solver and inference logic into an interactive Streamlit application, `sudoku_app.py`. Your application must implement the following:

**1. Puzzle selection & visual board display**
- An interactive selector/dropdown to pick any puzzle from `puzzles.json`.
- A visual rendering of the grid that clearly distinguishes the initial *givens* from empty cells.

**2. Full-grid auto-solver, with algorithm selection**
- A control (e.g. radio buttons) letting the user choose forward chaining (`solve_full_grid_fc`) or backward chaining (`solve_full_grid_bc`) before solving.
- A button that solves the full grid with the chosen algorithm, renders the solved state, and displays how long the solve took -- so the timing gap from task (d) is visible in the app, not just in the notebook.

**3. Targeted cell entailment query**
- Inputs for row ($r$), column ($c$), and value ($v$).
- A button that checks whether $\mathit{Is}_{rcv}$ is entailed (using `pl_bc_entails`) and displays the boolean verdict (`True` / `False`).

**4. Reasoning trace ("tutor mode")**
- Instrument your chosen inference algorithm (forward or backward chaining) to record the reasoning steps it takes while answering a query.
- Present that trace in a human-readable format -- not a raw Python string or internal symbol dictionary. For example: expandable cards/accordions showing the rule-firing sequence (*"Inferred $\mathit{Not}_{1,2,3}$ because row 1 already contains value 3"* $\implies$ *"Deduce $\mathit{Is}_{1,2,4}$ as the last remaining candidate"*), or plain-English sentences explaining each elimination by row, column, or box constraint.

Import `atom`, `build_definite_kb`, `build_general_kb`, `solve_full_grid_fc`, `solve_full_grid_bc`, and `pl_bc_entails` from `sudoku_solver.py`. Do not copy or rewrite these core functions in the Streamlit file. You may add app-specific helper functions where needed for the interface or reasoning trace.

Refer to the provided `StreamlitDeploymentGuide.pdf` guide to deploy your code and include the link for your Streamlit app below.


## Group Members

**Group Number: 17**

| Student Name | Student ID |
|---|---|
| Yang Yuxin | A0353937R |
| Gu Yuanding | A0352124R |
| Dong Yufei | A0357632Y |
| Shu Zixuan | A0353048E |


## Group Contributions

Our team divided the project into four main roles. Shu Zixuan developed the baseline implementation, including the Sudoku solver, notebook, and initial Streamlit interface. Gu Yuanding focused on debugging, fixing identified issues, and adding regression tests. Yang Yuxin was responsible for deployment, dependency configuration, and checking the application’s online functionality. Dong Yufei handled final integration, reviewed the project against the assignment requirements, and prepared the submission files.


## Submission

Submit only the following three files:

1. `Sudoku_Assignment.ipynb` — with all required cells run, outputs visible, and all conceptual questions answered.
2. `sudoku_solver.py` — containing your implementation of the knowledge-base and inference functions.
3. `sudoku_app.py` — containing your Streamlit application.

The provided support files are required to run the assignment but are not part of the student submission.

### Deployed Streamlit app URL

Paste your deployed Streamlit app URL here:

[Sudoku Logic Solver — deployed Streamlit app](https://sudoku-assignment-xznrjngyipqt7kg8px6tul.streamlit.app/)
